# 항만 ByT5 독립 파일럿 v1

L4 또는 A100 GPU를 선택한 뒤 위에서부터 실행하세요. 학습·평가 결과는 Drive에 저장됩니다. 전체 train 기반 33/36 모델을 검증 기준선으로 사용하지 않습니다. 공식 test 접근과 대회 제출은 하지 않습니다.

진행: 환경 설치 → Drive/입력 확인 → GPU 학습 → 기준 후보 → 조건부 점수·결합 평가. 학습은 기본 600 optimizer steps이며 소요 시간은 GPU에서 확인합니다.

In [ ]:
# 1. 필요한 라이브러리만 설치합니다. PyTorch는 코랩 기본 GPU 버전을 유지합니다.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate==1.9.0', 'safetensors', 'sentencepiece', 'lightgbm', 'rapidfuzz'])
print('설치 완료. 다음 셀을 실행하세요.')

In [ ]:
# 2. Drive 연결과 입력 검증
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import hashlib, zipfile, json, os
BASE = Path('/content/drive/MyDrive/항만_ByT5_파일럿_v1_20261027')
BASE.mkdir(parents=True, exist_ok=True)
ZIP = BASE / 'port_byt5_pilot_v1.zip'
if not ZIP.exists():
    print('준비된 port_byt5_pilot_v1.zip을 선택하세요.')
    uploaded = files.upload()
    ZIP.write_bytes(uploaded['port_byt5_pilot_v1.zip'])
assert hashlib.sha256(ZIP.read_bytes()).hexdigest() == '89a708690de5295fea575a673083c591c835aaddec9c0fa368ab9b7f9528c5d1', '다른 버전의 ZIP입니다.'
ROOT = Path('/content/port_byt5_pilot_v1')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for n in z.namelist():
        assert not Path(n).is_absolute() and '..' not in Path(n).parts
    z.extractall(ROOT)
os.environ['PYTHONHASHSEED']='0'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
sys.path.insert(0, str(ROOT))
from runtime_common import verify_package
manifest = verify_package(ROOT)
print({k: manifest[k] for k in ['training_pairs','training_sources','baseline_model_included','independent_eval_korean_vessels']})
subprocess.check_call([sys.executable, str(ROOT/'smoke_test.py')])
import torch
assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name())

In [ ]:
# 3. 실행 설정 (이름이 같으면 완료 결과 또는 마지막 체크포인트를 이어 씁니다.)
RUN_NAME = 'pilot_v1_s42_600'
STEPS = 600
BUILD_BASELINE_IF_MISSING = True  # 패키지에 기준선이 없으면 코랩 CPU에서 단일 시드 생성
RUN = BASE / 'runs' / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
print('결과 저장:', RUN)

In [ ]:
# 4. GPU 학습: 검증 선박을 학습에 쓰지 않으며 checkpoint 선택은 tune만 사용합니다.
subprocess.check_call([sys.executable, str(ROOT/'train_byt5.py'), '--root', str(ROOT), '--output', str(RUN), '--steps', str(STEPS)])
print(json.loads((RUN/'training_complete.json').read_text()))

In [ ]:
# 5. 자연 검색 상위 20후보. 정답을 강제로 추가하지 않습니다.
cmd = [sys.executable, str(ROOT/'evaluate_byt5.py'), 'baseline', '--root', str(ROOT), '--run', str(RUN)]
if BUILD_BASELINE_IF_MISSING: cmd.append('--allow-baseline-training')
subprocess.check_call(cmd)

In [ ]:
# 6. ByT5 조건부 점수와 실제 GPU 처리량을 계산합니다.
subprocess.check_call([sys.executable, str(ROOT/'evaluate_byt5.py'), 'score', '--root', str(ROOT), '--run', str(RUN)])

In [ ]:
# 7. 결합 가중은 tune에서만 고르고, 그 뒤 eval을 한 번 평가합니다.
subprocess.check_call([sys.executable, str(ROOT/'evaluate_byt5.py'), 'report', '--root', str(ROOT), '--run', str(RUN)])
import pandas as pd
m = pd.read_csv(RUN/'metrics.csv')
display(m[m.role.eq('eval')])
print(json.loads((RUN/'pilot_report.json').read_text()))
REPORT_ZIP = RUN / 'pilot_results.zip'
with zipfile.ZipFile(REPORT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in RUN.glob('*'):
        if p.suffix in ['.json','.jsonl','.csv']: z.write(p,p.name)
    for p in (RUN/'baseline').glob('*.json'): z.write(p,'baseline/'+p.name)
print('분석용 결과 ZIP:', REPORT_ZIP)
# 필요하면 아래 주석을 풀어 다운로드하세요.
# files.download(str(REPORT_ZIP))

학습 가중치는 Drive의 runs/RUN_NAME/best에 남습니다. 분석에는 pilot_results.zip을 전달하면 됩니다. 이것은 한글 52척의 독립 holdout 파일럿이며, 전체 OOF·Public 점수·제출 완료를 뜻하지 않습니다. GPU 실행 전에는 로컬 점검만 완료된 상태입니다.